In [ ]:
# sync_items_outsider.py
import os
import time
from datetime import datetime, timezone
from typing import Dict, Any, List

from pymongo import MongoClient, UpdateOne, ASCENDING

# ---- 可配置项 ----
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017/")
DB_NAME = os.getenv("DB_NAME", "inventory")
SRC_COLL = os.getenv("SRC_COLL", "items")
DST_COLL = os.getenv("DST_COLL", "items_outsider")

POLL_SECONDS = 300

# 汇率：CNY -> SGD（你给的是除以 5.46）
CNY_PER_SGD = float(os.getenv("CNY_PER_SGD", "5.46"))

# profit 压缩比例：压缩为 1/5
PROFIT_COMPRESS_RATIO = float(os.getenv("PROFIT_COMPRESS_RATIO", "5"))

# 是否把 tracking_number/note 置空
CLEAR_TRACKING_NUMBER = True
CLEAR_NOTE = True

# ---- 敏感关键词过滤 ----
FORBIDDEN_KEYWORDS = ["闲鱼", "验货宝"]

def clean_keywords(text: str) -> str:
    if not text:
        return ""
    cleaned = text
    for kw in FORBIDDEN_KEYWORDS:
        cleaned = cleaned.replace(kw, "")
    # 顺手清理多余空格
    return " ".join(cleaned.split())


# ---- 拼音转换（可选）----
def to_pinyin(name: str) -> str:
    name = (name or "").strip()
    if not name:
        return ""
    try:
        from pypinyin import lazy_pinyin  # type: ignore
        # “张三” -> "zhang san"
        return " ".join(lazy_pinyin(name))
    except Exception:
        # 没装 pypinyin 或出错，就原样返回（不中断同步）
        return name

def safe_int_round(x) -> int:
    if x is None:
        return 0
    try:
        return int(round(float(x)))
    except Exception:
        return 0



def transform_doc(src: Dict[str, Any]) -> Dict[str, Any]:
    doc = dict(src)

    # 1) BUY_IN -> CONSIGNMENT
    if doc.get("source_type") == "BUY_IN":
        doc["source_type"] = "CONSIGNMENT"
        doc["is_buy_in"] = False
        doc["is_consignment"] = True

    # 2) name：敏感词清洗
    doc["name"] = clean_keywords(doc.get("name", ""))

    # 3) seller_name：先清洗关键词，再转拼音
    seller_name_cleaned = clean_keywords(doc.get("seller_name", ""))
    doc["seller_name"] = to_pinyin(seller_name_cleaned)

    # 4) profit：压缩 + 换算 SGD
    profit = doc.get("profit", 0)
    profit_sgd = (float(profit or 0) / PROFIT_COMPRESS_RATIO) / CNY_PER_SGD
    doc["profit"] = safe_int_round(profit_sgd)
    doc["profit_currency"] = "SGD"

    # 5) cost：
    # - 未卖出：按原始 cost 换算 SGD（不压缩）
    # - 已卖出：cost = sold_price(SGD) - adjusted_profit(SGD)
    sold_price = doc.get("sold_price", None)

    if sold_price is None:
        raw_cost = doc.get("cost", 0)
        cost_sgd = float(raw_cost or 0) / CNY_PER_SGD
        doc["cost"] = safe_int_round(cost_sgd)
        doc["cost_currency"] = "SGD"
    else:
        sold_price_sgd = float(sold_price or 0)  # sold_price 永远是 SGD，不换汇
        adjusted_profit_sgd = float(doc.get("profit", 0))  # 已是 int(SGD)
        cost_sgd = sold_price_sgd - adjusted_profit_sgd

        # 如果你不想出现负数，打开这一行
        # cost_sgd = max(0.0, cost_sgd)

        doc["cost"] = safe_int_round(cost_sgd)
        doc["cost_currency"] = "SGD"

    # 6) tracking_number / note 清空（无条件）
    doc["tracking_number"] = ""
    doc["note"] = ""

    return doc



def main():
    client = MongoClient(MONGO_URI)
    db = client[DB_NAME]
    src = db[SRC_COLL]
    dst = db[DST_COLL]

    # 建议索引：按 sku 唯一同步更稳（如果你 sku 一定唯一）
    # 如果 sku 可能重复，就注释掉这行，仅用 _id 同步。
    dst.create_index([("sku", ASCENDING)], unique=True)

    print(f"[sync] {DB_NAME}.{SRC_COLL} -> {DB_NAME}.{DST_COLL}")
    print(f"[sync] poll every {POLL_SECONDS}s, CNY_PER_SGD={CNY_PER_SGD}, PROFIT_COMPRESS_RATIO={PROFIT_COMPRESS_RATIO}")

    while True:
        started = time.time()
        now = datetime.now(timezone.utc)

        # 读取源数据（你也可以加筛选：例如只同步 status != 'DELETED'）
        cursor = src.find({})

        ops: List[UpdateOne] = []
        count = 0

        for s in cursor:
            count += 1
            out = transform_doc(s)

            # upsert key：优先 sku（给店员看通常也用 sku），没有 sku 就退回 _id
            upsert_filter = {"sku": out["sku"]} if out.get("sku") else {"_id": out["_id"]}

            # 只更新字段（避免覆盖目标库里可能有的额外字段）
            # 同时保留目标库的 _id（如果用 sku upsert，Mongo 会生成新的 _id）
            out["updated_at"] = out.get("updated_at") or now  # 确保 required 字段存在
            ops.append(UpdateOne(upsert_filter, {"$set": out}, upsert=True))

            # 分批 bulk_write，避免一次太大
            if len(ops) >= 1000:
                dst.bulk_write(ops, ordered=False)
                ops.clear()

        if ops:
            dst.bulk_write(ops, ordered=False)

        elapsed = time.time() - started
        print(f"[sync] {now.isoformat()} processed={count} elapsed={elapsed:.2f}s")

        # 固定每分钟一次（考虑脚本执行时间）
        sleep_for = max(0, POLL_SECONDS - elapsed)
        time.sleep(sleep_for)

if __name__ == "__main__":
    main()


[sync] inventory.items -> inventory.items_outsider
[sync] poll every 300s, CNY_PER_SGD=5.46, PROFIT_COMPRESS_RATIO=5.0
[sync] 2025-12-20T09:49:29.680499+00:00 processed=5529 elapsed=1.45s
[sync] 2025-12-20T09:54:29.681638+00:00 processed=5529 elapsed=2.56s
[sync] 2025-12-20T09:59:29.683327+00:00 processed=5529 elapsed=1.91s
[sync] 2025-12-20T10:04:29.684574+00:00 processed=5529 elapsed=2.80s
[sync] 2025-12-20T10:09:29.685356+00:00 processed=5532 elapsed=2.72s
[sync] 2025-12-20T10:14:29.686379+00:00 processed=5538 elapsed=1.45s
[sync] 2025-12-20T10:19:29.687336+00:00 processed=5538 elapsed=2.75s
[sync] 2025-12-20T10:24:29.689115+00:00 processed=5538 elapsed=2.69s
[sync] 2025-12-20T10:29:29.689589+00:00 processed=5538 elapsed=1.43s
[sync] 2025-12-20T10:34:29.690923+00:00 processed=5538 elapsed=1.44s
[sync] 2025-12-20T10:39:29.692718+00:00 processed=5538 elapsed=1.32s
[sync] 2025-12-20T10:44:29.694484+00:00 processed=5539 elapsed=1.49s
[sync] 2025-12-20T10:49:29.695934+00:00 processed=553